# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/notebooks/03_working_with_the_full_release.ipynb)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


In [ ]:
%pip -q install duckdb huggingface_hub


In [3]:
import os, getpass

# CI and power users set HF_TOKEN in the environment; everyone else gets the safe prompt.
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


Paste your Hugging Face READ token (hf_...): ··········


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [4]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [5]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


clients with 12+ months of GSC history: 4


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


In [6]:
overall_end_date = con.sql(f"SELECT MAX(report_date) FROM {TABLES['fact_daily']}").fetchone()[0]
print(f"The overall end date (MAX report_date from fact_daily) is: {overall_end_date}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

The overall end date (MAX report_date from fact_daily) is: 2026-06-30


## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [5]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

111,247 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_3ffa76342f366962,content_bdc656fc8f037ac0,124.0,173.0,7.0,4.931609
1,client_3ffa76342f366962,content_315f75aa07a662bf,48.0,139.0,0.0,2.361667
2,client_e547b89c05043229,content_ded3d63f83e7a4cf,1795.0,588.0,4.0,9.341574
3,client_e547b89c05043229,content_b3a828afc221c27a,85.0,115.0,0.0,13.640051
4,client_e547b89c05043229,content_0252039a1f263e4e,299.0,352.0,0.0,34.854224


In [7]:
client_end_dates = con.sql(f"""
    SELECT client_hash_id, MAX(report_date) AS client_end_date
    FROM {TABLES['fact_daily']}
    GROUP BY client_hash_id
""").df()

clients_with_end_date = clients.merge(client_end_dates, on='client_hash_id', how='left')
print("Clients with their respective end dates (MAX report_date from fact_daily):")
display(clients_with_end_date.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Clients with their respective end dates (MAX report_date from fact_daily):


,client_hash_id,access_profile,gsc_data_start,ga4_data_start,client_end_date
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29,2026-06-30
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29,2026-06-30
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24,2026-06-30
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06,2026-06-30
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT,2026-06-30


In [8]:
clients_with_end_date['data_history_duration'] = clients_with_end_date['client_end_date'] - clients_with_end_date['gsc_data_start']
display(clients_with_end_date.head())

,client_hash_id,access_profile,gsc_data_start,ga4_data_start,client_end_date,data_history_duration
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29,2026-06-30,519 days
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29,2026-06-30,519 days
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24,2026-06-30,504 days
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06,2026-06-30,476 days
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT,2026-06-30,388 days


In [9]:
import pandas as pd

clients_less_than_90_days = clients_with_end_date[clients_with_end_date['data_history_duration'] < pd.Timedelta(days=90)]
display(clients_less_than_90_days.head())

,client_hash_id,access_profile,gsc_data_start,ga4_data_start,client_end_date,data_history_duration
57,client_8ddc46da5414ffd8,gsc_only,2026-04-07,NaT,2026-06-30,84 days
58,client_06d356715a8ff3b6,gsc_and_ga4,2026-04-10,2026-04-06,2026-06-30,81 days
59,client_0b245132bb722950,gsc_and_ga4,2026-04-12,2026-04-24,2026-06-30,79 days
60,client_9c26c096d6e57253,gsc_and_ga4,2026-04-29,2026-04-23,2026-06-30,62 days
61,client_c353557474475e51,gsc_and_ga4,2026-04-30,2026-06-01,2026-06-30,61 days


In [10]:
clients_with_missing_values = clients_with_end_date[clients_with_end_date.isnull().any(axis=1)]
display(clients_with_missing_values.head())

,client_hash_id,access_profile,gsc_data_start,ga4_data_start,client_end_date,data_history_duration
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT,2026-06-30,388 days
14,client_8ae2bfb5aa1ffa1e,gsc_only,2025-07-28,NaT,2026-06-30,337 days
15,client_8dbf3abdf07569e0,gsc_only,2025-07-29,NaT,2026-06-30,336 days
16,client_08a6a72ff48e62c0,gsc_only,2025-09-24,NaT,2026-06-30,279 days
22,client_795153d5b7850ccf,gsc_only,2025-09-24,NaT,2026-06-30,279 days


In [11]:
clients_to_exclude = pd.concat([
    clients_with_missing_values['client_hash_id'],
    clients_less_than_90_days['client_hash_id']
]).unique()

cleaned_clients = clients_with_end_date[
    ~clients_with_end_date['client_hash_id'].isin(clients_to_exclude)
]

print(f"Number of clients after cleaning: {len(cleaned_clients)}")
display(cleaned_clients.head())

Number of clients after cleaning: 41


,client_hash_id,access_profile,gsc_data_start,ga4_data_start,client_end_date,data_history_duration
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29,2026-06-30,519 days
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29,2026-06-30,519 days
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24,2026-06-30,504 days
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06,2026-06-30,476 days
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15,2026-06-30,377 days


In [ ]:
import pandas as pd

# Get the first client_hash_id from the cleaned_clients DataFrame as per user request
single_client_id = cleaned_clients['client_hash_id'].iloc[0]

print(f"Filtering fact_daily for client: {single_client_id}")

# Use DuckDB to filter fact_daily for this single client_hash_id
# This approach avoids the large IN clause or JOIN for many clients, focusing on one.
fact_daily_cleaned = con.sql(f"""
    SELECT *
    FROM {TABLES['fact_daily']}
    WHERE client_hash_id = '{single_client_id}';
""").df()

print(f"Number of rows in fact_daily for {single_client_id} after cleaning: {len(fact_daily_cleaned):,}")
display(fact_daily_cleaned.head())

Filtering fact_daily for client: client_9958f0a7ae1df715


## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [ ]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))


Whatever number you just got: interrogate it before you believe it. Which feature carries the
signal? Does it survive a per-client split (train on some clients, test on others)? That
question — *does it generalize across clients?* — is exactly what separates a capstone-grade
result from a lucky split.

## Your turn

1. Re-run section 3 with a **90-day** window and a `HAVING` threshold of your choice.
2. Add one feature you believe in (position volatility? weekend share? query concentration?).
3. Replace the random split with **GroupShuffleSplit on `client_hash_id`** and compare.

## Working locally instead

```python
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])
```
Then point `REL` at that local path. Download only the month partitions you need — the
`allow_patterns` filter above is the whole trick.

---

**Where this fits:** every lane brief assumes you can produce per-content feature tables like
the one you just built. The lane datasets under the `lanes` HF repo are pre-cut examples of
exactly this pattern — but for the capstone, features you engineered yourself from the full
release beat any pre-cut file.
